# Analisis Sentimen & Pre-Processing Ulasan Aplikasi MyTelkomsel (Google Play Store)

**Mata Kuliah:** Machine Learning & Big Data (Semester 5)  
**Sumber Data:** Google Play Store (`com.telkomsel.telkomselcm`)  
**Target Dataset:** 10.000 Ulasan Terbaru (*Natural Sampling*)  
**Pipeline Terperbarui:** Scraping $\rightarrow$ Screening (Kosong, Duplikat Persis, <3 Kata, Near-Duplicate) $\rightarrow$ Dua Jalur Pra-Proses (Klasik/TF-IDF & IndoBERT) $\rightarrow$ Validasi Label (Audit 150 Sampel + Sentimen Leksikon) $\rightarrow$ N-gram Pra-Stopword Berbatas Klausa, Log-Odds Ratio, Analisis Aspek, Word Cloud (`color_func` + Sensor Kata Kasar + Log-Odds) $\rightarrow$ Stratified Split 80/20 & Persiapan Modeling (`macro-F1`)


---
## 1. Instalasi Library Pendukung (Google Colab Environment)
Memasang pustaka untuk *scraping*, NLP bahasa Indonesia (*PySastrawi*, *NLTK*), konversi emoji (*emoji*), analisis TF-IDF/ML (*scikit-learn*), serta visualisasi (*WordCloud*, *Matplotlib*, *Seaborn*).


In [ ]:
# Alasan (A.6): Tambahkan pustaka 'emoji' untuk konversi emoji ke teks sentimen.
!pip install -q google-play-scraper PySastrawi nltk wordcloud matplotlib seaborn tqdm emoji scikit-learn


---
## 2. Import Library dan Konfigurasi Pipeline
Sel ini memuat seluruh dependensi dan parameter konfigurasi (whitelist negasi yang mudah diedit, kata kontras/penguat, penggantian frasa regex, kamus emoji, normalisasi kata Inggris, filter kata kasar khusus Word Cloud, leksikon sentimen, serta kamus aspek).


In [ ]:
import json
import os
import re
import string
import time
import zipfile
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

# Scraping Google Play Store
from google_play_scraper import Sort, reviews

# NLP Bahasa Indonesia
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Visualisasi Kata
from wordcloud import WordCloud

# Alasan (A.6): Gunakan pustaka emoji jika tersedia, dengan fallback kamus bawaan
try:
    import emoji
    HAS_EMOJI_LIB = True
except ImportError:
    HAS_EMOJI_LIB = False

for path, name in [('tokenizers/punkt', 'punkt'), ('tokenizers/punkt_tab', 'punkt_tab'), ('corpora/stopwords', 'stopwords')]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(name, quiet=True)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
%matplotlib inline

# ─── Konfigurasi Domain & Aturan NLP ───────────────────────────────────────────

# Kata negasi yang boleh bergabung dengan kata bermakna
NEGATION_WORDS = {'tidak', 'belum', 'bukan', 'jangan', 'tanpa', 'tak', 'tiada'}

# Alasan (A.5): Whitelist kata bermakna agar negasi tidak menempel pada stopword
# (menghindari token semu seperti 'jangan_di', 'tidak_pernah', 'tidak_usah').
# Daftar ini mudah diedit sesuai kebutuhan analisis.
NEGATION_TARGET_WHITELIST = {
    'bisa', 'bagus', 'jelas', 'stabil', 'sesuai', 'lancar', 'puas', 'mau', 'ada',
    'masuk', 'guna', 'berguna', 'suka', 'paham', 'benar', 'wajar', 'layak', 'normal',
    'aman', 'berfungsi', 'fungsi', 'muncul', 'kirim', 'terkirim', 'buka', 'terbuka',
    'jalan', 'berjalan', 'connect', 'konek', 'nyambung', 'terhubung', 'respon',
    'responsif', 'cepat', 'murah', 'mahal', 'worth', 'rekomen', 'ramah', 'peduli',
    'tahu', 'mengerti', 'sebanding', 'berubah', 'hilang', 'rusak', 'lemot', 'lambat',
    'error', 'gangguan', 'ribet', 'susah', 'sulit', 'enak', 'nyaman', 'mudah',
    'berhasil', 'mampu', 'sempat', 'sanggup', 'terima', 'beli', 'pakai', 'bayar',
}

# Alasan (A.1 & A.2): Kata kontras dipakai sebagai batas klausa n-gram agar tidak muncul
# bigram semu seperti 'mahal sinyal' dari 'mahal tapi sinyal jelek', dan tidak dihapus.
CONTRAST_WORDS = {'tapi', 'tetapi', 'padahal', 'namun', 'sedangkan', 'kecuali', 'melainkan'}

# Alasan (A.2): Kata penguat/pelemah menentukan intensitas sentimen sehingga dipertahankan.
INTENSIFIER_WORDS = {
    'sangat', 'banget', 'kurang', 'sekali', 'terlalu', 'paling',
    'agak', 'cukup', 'lebih', 'malah', 'justru', 'amat',
}
CONTRAST_AND_INTENSIFIER_WORDS = CONTRAST_WORDS.union(INTENSIFIER_WORDS)

# Alasan (B.2): Entri berspasi dipindahkan dari kamus_slang.json ke daftar penggantian
# frasa berbasis regex yang dieksekusi SEBELUM tokenizing.
PHRASE_REPLACEMENTS = {
    'kayak': 'seperti',
    'force close': 'keluar_paksa',
    'blank screen': 'layar_putih',
    'customer service': 'cs',
    'nge lag': 'lag',
    'nge-lag': 'lag',
    'muter muter': 'loading',
    'terima kasih': 'terima_kasih',
    'no signal': 'tidak ada sinyal',
    'bad network': 'jaringan buruk',
    'slow connection': 'koneksi lambat',
    'worth it': 'sebanding',
    'pulsa safe': 'pulsa_safe',
}

# Alasan (A.6): Konversi emoji ke teks sentimen sebelum non-ASCII dihapus.
EMOJI_SENTIMENT_MAP = {
    '😡': ' marah ', '🤬': ' marah ', '😠': ' marah ', '😤': ' marah ',
    '👎': ' buruk ', '💩': ' buruk ', '🤮': ' muak ', '🤢': ' muak ',
    '😭': ' sedih ', '😢': ' sedih ', '😔': ' sedih ', '😞': ' sedih ', '🥺': ' sedih ', '🥲': ' sedih ',
    '😑': ' kecewa ', '😒': ' kecewa ', '🤦': ' kecewa ', '🙄': ' kecewa ', '💔': ' kecewa ',
    '👍': ' bagus ', '⭐': ' bagus ', '🌟': ' bagus ', '🔥': ' mantap ', '💯': ' mantap ',
    '😁': ' senang ', '😄': ' senang ', '😊': ' senang ', '🙂': ' senang ', '😇': ' senang ',
    '🥰': ' suka ', '😍': ' suka ', '❤️': ' suka ', '❤': ' suka ', '💕': ' suka ', '💖': ' suka ',
    '😂': ' ketawa ', '🤣': ' ketawa ', '🙏': ' mohon ',
}

# Alasan (A.7): Normalisasi kata Inggris bermakna ke padanan bahasa Indonesia.
ENGLISH_TO_INDONESIAN = {
    'good': 'bagus', 'best': 'terbaik', 'nice': 'bagus', 'great': 'bagus',
    'awesome': 'mantap', 'cool': 'keren', 'fast': 'cepat', 'slow': 'lambat',
    'bad': 'buruk', 'worst': 'terburuk', 'poor': 'buruk', 'trash': 'buruk',
    'garbage': 'buruk', 'useless': 'tidak berguna', 'worthless': 'tidak berguna',
    'helpful': 'membantu', 'easy': 'mudah', 'hard': 'sulit', 'expensive': 'mahal',
    'cheap': 'murah', 'love': 'suka', 'like': 'suka', 'hate': 'benci',
    'happy': 'senang', 'disappointed': 'kecewa', 'fix': 'perbaiki', 'fixed': 'perbaiki',
    'broken': 'rusak', 'crash': 'keluar paksa', 'freeze': 'macet', 'stuck': 'macet',
    'smooth': 'lancar', 'stable': 'stabil', 'connection': 'koneksi', 'network': 'jaringan',
    'price': 'harga', 'service': 'layanan', 'customer': 'pelanggan', 'quota': 'kuota',
    'money': 'uang', 'free': 'gratis', 'please': 'tolong', 'sorry': 'maaf',
    'thank': 'terima_kasih', 'recommend': 'rekomendasi', 'recommended': 'rekomendasi',
    'not': 'tidak', 'very': 'sangat', 'but': 'tapi',
}

# Alasan (A.7): Stopword dasar bahasa Inggris (tanpa 'no' karena di ID sering berarti 'nomor').
ENGLISH_BASIC_STOPWORDS = {
    'the', 'is', 'am', 'are', 'was', 'were', 'be', 'been', 'being',
    'a', 'an', 'of', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'from',
    'as', 'it', 'its', 'this', 'that', 'these', 'those', 'my', 'your',
    'you', 'me', 'we', 'our', 'they', 'their', 'he', 'she', 'and', 'or',
    'so', 'if', 'when', 'while', 'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'can', 'could', 'should', 'boss', 'bos', 'bro', 'guys',
    'sir', 'dear', 'gan', 'cuy', 'broo',
}

# Alasan (A.4): Kata bertanda hubung leksikal yang dilindungi agar tidak menjadi 'tibatiba'.
LEXICALIZED_HYPHEN_WORDS = {
    'tiba-tiba', 'gara-gara', 'sia-sia', 'rata-rata', 'masing-masing',
    'sehari-hari', 'm-banking', 'e-wallet', 'e-commerce', 'non-paket',
    'anti-sedot', 'pulsa-safe',
}

# Alasan (A.2): Kata dalam stopword yang ditandai memiliki nuansa sentimen/pragmatik.
SENTIMENT_FLAGGED_STOPS = {
    'terima', 'semoga', 'moga', 'maaf', 'ampun', 'tolong', 'mohon', 'bantu',
    'baik', 'benar', 'salah', 'masalah', 'lama', 'butuh', 'mending', 'hadehh',
    'hadeh', 'jangan_mahal', 'tidak_ngotak', 'tidak_masuk', 'cukup', 'biasa',
}

# Kata domain yang tidak di-stem oleh Sastrawi
STEM_EXCEPTIONS = {
    'paket', 'jaringan', 'sinyal', 'kuota', 'pulsa', 'internet',
    'aplikasi', 'login', 'loading', 'verifikasi', 'notifikasi',
    'transaksi', 'pembayaran', 'pelayanan', 'layanan', 'berlangganan', 'langganan', 'registrasi',
    'promo', 'bonus', 'diskon', 'tagihan', 'pascabayar', 'prabayar',
    'provider', 'operator', 'server', 'error', 'bug', 'lag', 'hang',
    'otp', 'restart', 'download', 'upload', 'streaming',
    'tiba-tiba', 'gara-gara', 'sia-sia', 'rata-rata', 'masing-masing',
    'sehari-hari', 'm-banking', 'e-wallet', 'e-commerce', 'non-paket',
    'terima_kasih', 'keluar_paksa', 'pulsa_safe',
    'jelas', 'kualitas', 'gratis', 'stabil', 'malas', 'puas',
    'habis', 'bagus', 'cepat', 'lambat', 'murah', 'mahal',
    'aktif', 'lancar', 'sesuai', 'guna', 'masuk',
    'boros', 'parah', 'susah', 'sulit', 'gagal', 'hilang',
    'kecewa', 'rugi', 'busuk', 'potong', 'hangus',
    'pelanggan', 'dipakai', 'telepon',
}

# Alasan (D.1): Sensor kata kasar HANYA pada gambar word cloud (tidak mengubah data/teks_final).
PROFANITY_FILTER = {
    'tolol', 'babi', 'anjing', 'anj', 'ajg', 'asu', 'asuu', 'bangsat', 'bangke',
    'bangkee', 'kontol', 'memek', 'ngentot', 'ngentod', 'tai', 'taik', 'thai',
    'goblok', 'goblog', 'bodoh', 'idiot', 'bajingan', 'brengsek', 'setan', 'iblis',
    'jancok', 'cuk', 'cok', 'bgst', 'kntl', 'mmk', 'ngtd', 'pantek', 'puki',
    'kimak', 'sialan', 'keparat', 'bacot', 'cot', 'bejat', 'ngotak',
}

NEGATIVE_LEXICON = {
    'mahal', 'error', 'bug', 'lag', 'hang', 'lambat', 'lelet', 'berat',
    'susah', 'sulit', 'kecewa', 'buruk', 'jelek', 'parah', 'payah',
    'gagal', 'rusak', 'hancur', 'mati', 'hilang', 'hangus', 'potong',
    'sedot', 'boros', 'rugi', 'sayang', 'busuk', 'ampas', 'sampah',
    'loading', 'aneh', 'ribet', 'rumit', 'ganggu', 'gangguan',
    'salah', 'turun', 'keluh', 'kendala', 'ilang',
    'kikir', 'habis', 'turunin', 'naikin', 'sumpah', 'gara', 'iklan',
    'kalah', 'pindah', 'kacau', 'bobrok', 'burik', 'bapuk', 'muak',
    'emosi', 'marah', 'sedih', 'sesal', 'korup', 'korupsi', 'kemahalan',
    'terburuk', 'merugikan', 'mengecewakan', 'bermasalah', 'macet', 'benci',
    'pelit', 'sia-sia', 'keluar_paksa',
    'tidak_bisa', 'tidak_bagus', 'tidak_jelas', 'tidak_stabil', 'tidak_sesuai',
    'tidak_guna', 'tidak_berguna', 'tidak_masuk', 'tidak_lancar', 'tidak_puas',
    'tidak_mau', 'tidak_ada', 'tidak_normal', 'tidak_aman', 'tidak_berfungsi',
    'belum_bisa', 'belum_stabil', 'belum_sesuai', 'belum_lancar',
    'moga', 'tolong', 'mohon', 'sisa', 'cek', 'jaring', 'ngeleg',
    'kesal', 'kesel', 'bangkrut', 'spam', 'restart', 'paksa',
    'hapus', 'darurat', 'putus', 'gila', 'lupa',
}

POSITIVE_LEXICON = {
    'bagus', 'mantap', 'keren', 'lancar', 'cepat', 'mudah', 'murah',
    'puas', 'senang', 'suka', 'best', 'gratis', 'stabil', 'praktis',
    'nyaman', 'setia', 'percaya', 'manfaat', 'bermanfaat', 'reward', 'hadiah',
    'terima_kasih', 'terimakasih', 'terbaik', 'membantu', 'hemat', 'terjangkau',
    'lumayan', 'good', 'top', 'mantep', 'aman', 'untung', 'sesuai', 'super',
    'rekomendasi', 'responsif', 'ramah', 'berhasil', 'sebanding',
}

NEUTRAL_TOPIC_WORDS = {
    'paket', 'kuota', 'jaringan', 'sinyal', 'internet', 'pulsa', 'data',
    'kartu', 'harga', 'nomor', 'provider', 'operator', 'simpati', 'indihome',
    'wifi', 'youtube', 'game', 'sms', 'daerah', 'akun', 'sistem', 'fitur',
    'beli', 'pakai', 'isi', 'buka', 'bayar', 'pembayaran', 'transaksi',
    'update', 'login', 'akses', 'aktif', 'ganti', 'tukar', 'cek', 'sisa',
    'kirim', 'ambil', 'jual', 'poin', 'saldo', 'tagihan', 'registrasi',
    'layanan', 'langganan', 'langgan', 'layan', 'koneksi', 'metode',
}

# Alasan (D.3): Kamus kata kunci untuk analisis aspek berbasis leksikon.
ASPECT_KEYWORDS = {
    'Aplikasi': {
        'login', 'error', 'update', 'bug', 'loading', 'otp', 'keluar_paksa',
        'keluar', 'masuk', 'daftar', 'registrasi', 'buka', 'verifikasi',
        'hang', 'macet', 'restart', 'layar', 'crash', 'fitur', 'menu',
        'notifikasi', 'akun', 'aplikasi',
    },
    'Jaringan': {
        'sinyal', 'jaringan', 'lambat', 'lag', 'koneksi', 'internet', 'lemot',
        'lelet', 'putus', 'gangguan', 'hilang', 'stabil', 'cepat', 'lancar',
        'tower', '4g', '5g',
    },
    'Harga/Paket': {
        'mahal', 'kuota', 'pulsa', 'paket', 'harga', 'murah', 'promo',
        'diskon', 'potong', 'sedot', 'hangus', 'boros', 'bayar', 'saldo',
        'tagihan', 'beli', 'transaksi', 'poin', 'voucher', 'gratis',
    },
    'Layanan': {
        'cs', 'tolong', 'bantu', 'respon', 'layanan', 'pelayanan', 'admin',
        'keluhan', 'komplain', 'solusi', 'perbaiki', 'perbaikan', 'ramah',
        'peduli', 'lapor', 'laporan',
    },
}

# Kamus slang tersinkronisasi dengan kamus_slang.json (dapat dimuat dari file atau fallback inline)
DEFAULT_SLANG_DICT = {
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "sdh": "sudah",
    "udh": "sudah",
    "bgt": "banget",
    "bgtu": "begitu",
    "bgmn": "bagaimana",
    "gmn": "bagaimana",
    "gk": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "ga": "tidak",
    "tdk": "tidak",
    "tak": "tidak",
    "kaga": "tidak",
    "kagak": "tidak",
    "engga": "tidak",
    "enggak": "tidak",
    "ngk": "tidak",
    "blm": "belum",
    "blom": "belum",
    "krn": "karena",
    "karna": "karena",
    "tp": "tapi",
    "tpi": "tapi",
    "kalo": "kalau",
    "kl": "kalau",
    "klo": "kalau",
    "jd": "jadi",
    "jdi": "jadi",
    "dr": "dari",
    "dri": "dari",
    "sy": "saya",
    "gw": "saya",
    "gue": "saya",
    "gua": "saya",
    "aku": "saya",
    "km": "kamu",
    "lu": "kamu",
    "loe": "kamu",
    "lo": "kamu",
    "dpt": "dapat",
    "bs": "bisa",
    "bisaa": "bisa",
    "bkn": "bukan",
    "bknnya": "bukan",
    "skrg": "sekarang",
    "skrang": "sekarang",
    "skrng": "sekarang",
    "uda": "sudah",
    "bangat": "banget",
    "baguss": "bagus",
    "mahall": "mahal",
    "trimakasih": "terima kasih",
    "gajelas": "tidak jelas",
    "gakjelas": "tidak jelas",
    "gada": "tidak ada",
    "kualitasnya": "kualitas",
    "jelasnya": "jelas",
    "abis": "habis",
    "nomer": "nomor",
    "tau": "tahu",
    "smp": "sampai",
    "sampe": "sampai",
    "trs": "terus",
    "trus": "terus",
    "lg": "lagi",
    "lgi": "lagi",
    "aj": "saja",
    "aja": "saja",
    "sm": "sama",
    "kek": "seperti",
    "kyk": "seperti",
    "bnr": "benar",
    "bener": "benar",
    "beneran": "benar",
    "emg": "memang",
    "emang": "memang",
    "msh": "masih",
    "byk": "banyak",
    "bnyk": "banyak",
    "bbrp": "beberapa",
    "lbh": "lebih",
    "pdhl": "padahal",
    "pd": "pada",
    "knp": "kenapa",
    "knapa": "kenapa",
    "jg": "juga",
    "hrs": "harus",
    "hrus": "harus",
    "dlm": "dalam",
    "tlg": "tolong",
    "mhn": "mohon",
    "makasi": "terima kasih",
    "makasih": "terima kasih",
    "thx": "terima kasih",
    "thanks": "terima kasih",
    "tsel": "telkomsel",
    "mytelkomsel": "telkomsel",
    "mytsel": "telkomsel",
    "apk": "aplikasi",
    "app": "aplikasi",
    "apps": "aplikasi",
    "apikasi": "aplikasi",
    "apknya": "aplikasi",
    "lemot": "lambat",
    "lemott": "lambat",
    "lola": "lambat",
    "lelet": "lambat",
    "loding": "loading",
    "load": "loading",
    "loadingnya": "loading",
    "muter": "loading",
    "eror": "error",
    "erorr": "error",
    "err": "error",
    "ngebug": "bug",
    "forceclose": "keluar_paksa",
    "fc": "keluar_paksa",
    "kuotany": "kuota",
    "kuotanya": "kuota",
    "kuotaa": "kuota",
    "kouta": "kuota",
    "pulsany": "pulsa",
    "pulsanya": "pulsa",
    "plsa": "pulsa",
    "sinyalny": "sinyal",
    "sinyalnya": "sinyal",
    "signal": "sinyal",
    "jaringanny": "jaringan",
    "jaringanya": "jaringan",
    "jaringannya": "jaringan",
    "internetnya": "internet",
    "promonya": "promo",
    "paketnya": "paket",
    "paketannya": "paket",
    "harganya": "harga",
    "datanya": "data",
    "aplikasinya": "aplikasi",
    "loginnya": "login",
    "verif": "verifikasi",
    "notif": "notifikasi",
    "pake": "pakai",
    "pakek": "pakai",
    "dipake": "dipakai",
    "nelpon": "telepon",
    "ilang": "hilang",
    "benerin": "perbaiki",
    "kasi": "kasih",
    "sya": "saya",
    "cepet": "cepat",
    "tetep": "tetap",
    "ngeleg": "lag",
    "ngelek": "lag",
    "terimakasih": "terima kasih",
    "gini": "begini",
    "ngak": "tidak",
    "bngt": "banget",
    "yaa": "ya",
    "nihh": "",
    "nyedot": "sedot",
    "murmer": "murah",
    "mumer": "murah",
    "mantep": "mantap",
    "jos": "mantap",
    "gabisa": "tidak bisa",
    "gasuka": "tidak suka",
    "gatau": "tidak tahu",
    "gaada": "tidak ada",
    "gamau": "tidak mau",
    "gabener": "tidak benar",
    "gapaham": "tidak paham",
    "ngecek": "cek",
    "ngelag": "lag",
    "ngehang": "hang",
    "nyesel": "sesal",
    "ngebuka": "buka",
    "ngerestart": "restart",
    "lemooott": "lambat",
    "ampas": "buruk",
    "sampah": "buruk",
    "jelek": "buruk",
    "rewel": "bermasalah",
    "ribet": "rumit",
    "ruwet": "rumit",
    "jgn": "jangan",
    "liat": "lihat",
    "dapet": "dapat",
    "ngisi": "isi",
    "nunggu": "tunggu",
    "gampang": "mudah",
    "memuaskan": "puas",
    "bgus": "bagus",
    "make": "pakai",
    "memakai": "pakai",
    "terkadang": "kadang",
    "berlangganan": "langganan",
    "melayani": "layanan",
    "pelayanan": "layanan",
    "moga": "semoga",
    "kesel": "kesal",
    "mantab": "mantap"
}

if os.path.exists('kamus_slang.json'):
    with open('kamus_slang.json', 'r', encoding='utf-8') as f:
        SLANG_DICT = {k: v for k, v in json.load(f).items() if ' ' not in k and k != v}
    print(f"📖 Kamus slang dimuat dari 'kamus_slang.json' ({len(SLANG_DICT)} entri).")
else:
    SLANG_DICT = {k: v for k, v in DEFAULT_SLANG_DICT.items() if ' ' not in k and k != v}
    with open('kamus_slang.json', 'w', encoding='utf-8') as f:
        json.dump(SLANG_DICT, f, ensure_ascii=False, indent=4)
    print(f"📖 Menggunakan kamus slang bawaan tersinkron ({len(SLANG_DICT)} entri) & disimpan ke 'kamus_slang.json'.")

print("✅ Seluruh library dan konfigurasi siap!")


---
## 3. Pengumpulan Data (Scraping Ulasan Google Play Store)
Mengambil **10.000 ulasan terbaru** aplikasi MyTelkomsel (`com.telkomsel.telkomselcm`). Jika `ulasan_mentah.csv` sudah ada di direktori kerja (misal diunggah ke Colab untuk reproduksibilitas), Anda dapat langsung memakainya dengan mengatur `USE_EXISTING_RAW_CSV = True`.


In [ ]:
APP_ID = 'com.telkomsel.telkomselcm'
TARGET_REVIEWS = 10000
BATCH_SIZE = 200
USE_EXISTING_RAW_CSV = False  # Ubah ke True jika ingin memuat ulasan_mentah.csv yang sudah ada

if USE_EXISTING_RAW_CSV and os.path.exists('ulasan_mentah.csv'):
    df_raw = pd.read_csv('ulasan_mentah.csv')
    print(f"📂 Memuat dataset mentah yang sudah ada: {len(df_raw):,} ulasan.")
else:
    print(f"🚀 Memulai scraping {TARGET_REVIEWS:,} ulasan terbaru untuk '{APP_ID}'...")
    all_reviews = []
    continuation_token = None
    batch_num = 1
    start_time = time.time()

    while len(all_reviews) < TARGET_REVIEWS:
        count_to_fetch = min(BATCH_SIZE, TARGET_REVIEWS - len(all_reviews))
        try:
            result, continuation_token = reviews(
                APP_ID, lang='id', country='id', sort=Sort.NEWEST,
                count=count_to_fetch, continuation_token=continuation_token
            )
            if not result:
                print("⚠️ Tidak ada ulasan tambahan yang dapat diambil.")
                break
            all_reviews.extend(result)
            elapsed = time.time() - start_time
            print(f"Batch {batch_num:02d}: +{len(result)} ulasan | Total: {len(all_reviews):,}/{TARGET_REVIEWS:,} ({elapsed:.1f} dtk)")
            batch_num += 1
            if not continuation_token:
                break
            time.sleep(0.5)
        except Exception as e:
            print(f"❌ Error pada batch {batch_num}: {e}")
            time.sleep(2)

    df_raw = pd.DataFrame(all_reviews)
    cols = ['content', 'score', 'at', 'appVersion', 'thumbsUpCount']
    df_raw = df_raw[[c for c in cols if c in df_raw.columns]]
    df_raw.to_csv('ulasan_mentah.csv', index=False, encoding='utf-8')
    print(f"✅ Scraping selesai! {len(df_raw):,} ulasan tersimpan di 'ulasan_mentah.csv'.")

df_raw.head()


---
## 4. Data Screening (Termasuk Near-Duplicate) & Pelabelan Sentimen
Tahapan screening mencatat jumlah data pada setiap tahap dan menyimpannya ke `ringkasan_screening.csv`:
1. **Mentah $\rightarrow$ Kosong (Null/Empty)**
2. **Duplikat Persis**
3. **Terlalu Pendek (< 3 kata)**
4. **Near-Duplicate** *(hash teks ternormalisasi + TF-IDF cosine similarity > 0,90)*
5. **Preservasi kolom `at` (`tanggal_ulasan`) dan `appVersion`** untuk analisis tren waktu & versi aplikasi.


In [ ]:
def detect_near_duplicates(df, text_col='content', cosine_threshold=0.90, block_size=500):
    # Alasan (A.8): Menangkap ulasan hampir sama (beda kapitalisasi/tanda baca/1 kata)
    # dengan hash ternormalisasi lalu TF-IDF cosine > 0.90 secara bertahap (hemat RAM).
    def _norm_hash(text):
        s = str(text).lower()
        s = re.sub(r'[^\w\s]', '', s)
        return re.sub(r'\s+', ' ', s).strip()

    norm_texts = df[text_col].apply(_norm_hash)
    seen_hashes = set()
    keep_mask_hash = []
    for nh in norm_texts:
        if nh in seen_hashes:
            keep_mask_hash.append(False)
        else:
            seen_hashes.add(nh)
            keep_mask_hash.append(True)

    df_hash = df[keep_mask_hash].reset_index(drop=True)
    hash_dropped = len(df) - len(df_hash)

    norm_list = df_hash[text_col].apply(_norm_hash).tolist()
    tfidf_dropped = 0
    if len(norm_list) > 1:
        vec = TfidfVectorizer(min_df=1)
        X = vec.fit_transform(norm_list)
        n = X.shape[0]
        to_drop = set()
        for start in range(0, n, block_size):
            end = min(start + block_size, n)
            sim_block = (X[start:end] @ X.T).toarray()
            for i_rel in range(end - start):
                i = start + i_rel
                if i in to_drop:
                    continue
                matches = np.where(sim_block[i_rel, i + 1:] > cosine_threshold)[0]
                for m in matches:
                    to_drop.add(i + 1 + int(m))
        keep_mask = [idx not in to_drop for idx in range(n)]
        tfidf_dropped = len(to_drop)
        df_res = df_hash[keep_mask].reset_index(drop=True)
    else:
        df_res = df_hash

    return df_res, hash_dropped, tfidf_dropped


raw_count = len(df_raw)

# 1. Hapus kosong/null
df_screened = df_raw.dropna(subset=['content']).copy()
df_screened = df_screened[df_screened['content'].astype(str).str.strip() != ''].copy()
null_count = raw_count - len(df_screened)
after_null = len(df_screened)

# 2. Hapus duplikat persis
df_screened = df_screened.drop_duplicates(subset=['content']).copy()
dedup_count = after_null - len(df_screened)
after_dedup = len(df_screened)

# 3. Hapus < 3 kata
df_screened['word_count'] = df_screened['content'].apply(lambda x: len(str(x).strip().split()))
df_screened = df_screened[df_screened['word_count'] >= 3].reset_index(drop=True)
short_count = after_dedup - len(df_screened)
after_short = len(df_screened)

# 4. Hapus near-duplicate
df_screened, hash_dup_count, tfidf_dup_count = detect_near_duplicates(df_screened)
near_dup_count = hash_dup_count + tfidf_dup_count
after_near_dup = len(df_screened)

# Alasan (A.9): Simpan kolom tanggal ulasan dan appVersion untuk analisis tren
if 'at' in df_screened.columns:
    df_screened['at'] = pd.to_datetime(df_screened['at'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
    df_screened['tanggal_ulasan'] = df_screened['at'].astype(str).str.slice(0, 10)
else:
    df_screened['at'] = ''
    df_screened['tanggal_ulasan'] = ''

if 'appVersion' in df_screened.columns:
    df_screened['appVersion'] = df_screened['appVersion'].fillna('Tidak Diketahui').replace('', 'Tidak Diketahui')
else:
    df_screened['appVersion'] = 'Tidak Diketahui'

# 5. Pelabelan Sentimen
def assign_sentiment(score):
    if score in [1, 2]:
        return 'Negatif'
    elif score == 3:
        return 'Netral'
    elif score in [4, 5]:
        return 'Positif'
    return 'Lainnya'

df_screened['sentiment'] = df_screened['score'].apply(assign_sentiment)
df_screened.to_csv('ulasan_screening.csv', index=False, encoding='utf-8')

# Alasan (A.10): Catat ringkasan tiap tahap screening dan ekspor ke CSV
df_screening_summary = pd.DataFrame([
    {'Tahap': '1. Data Mentah (Raw)', 'Dibuang': 0, 'Sisa': raw_count, 'Persentase_Sisa': 100.0, 'Keterangan': 'Hasil scraping awal'},
    {'Tahap': '2. Hapus Kosong (Null/Empty)', 'Dibuang': null_count, 'Sisa': after_null, 'Persentase_Sisa': round(after_null / raw_count * 100, 2), 'Keterangan': 'Teks kosong/NaN'},
    {'Tahap': '3. Hapus Duplikat Persis', 'Dibuang': dedup_count, 'Sisa': after_dedup, 'Persentase_Sisa': round(after_dedup / raw_count * 100, 2), 'Keterangan': 'Teks identik persis'},
    {'Tahap': '4. Hapus Terlalu Pendek (<3 kata)', 'Dibuang': short_count, 'Sisa': after_short, 'Persentase_Sisa': round(after_short / raw_count * 100, 2), 'Keterangan': 'Minim konteks'},
    {'Tahap': '5. Hapus Near-Duplicate (Cosine > 0.9)', 'Dibuang': near_dup_count, 'Sisa': after_near_dup, 'Persentase_Sisa': round(after_near_dup / raw_count * 100, 2), 'Keterangan': f'Hash norm: {hash_dup_count}, TF-IDF cosine > 0.9: {tfidf_dup_count}'},
])
df_screening_summary.to_csv('ringkasan_screening.csv', index=False, encoding='utf-8')

print("📋 RINGKASAN SCREENING PER TAHAP:")
display(df_screening_summary)
print("\n📊 Distribusi Sentimen:")
print(df_screened['sentiment'].value_counts())


In [ ]:
# Visualisasi Distribusi Rating dan Sentimen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
score_counts = df_screened['score'].value_counts().sort_index()
sns.barplot(x=score_counts.index, y=score_counts.values, ax=axes[0], palette='Blues_r')
axes[0].set_title('Distribusi Rating Bintang (1 - 5)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Jumlah Ulasan')
for i, v in enumerate(score_counts.values):
    axes[0].text(i, v + (max(score_counts.values) * 0.015), f"{v:,}\n({v/len(df_screened)*100:.1f}%)", ha='center', fontsize=9)

sentiment_order = ['Negatif', 'Netral', 'Positif']
sent_counts = df_screened['sentiment'].value_counts().reindex(sentiment_order)
sns.barplot(x=sent_counts.index, y=sent_counts.values, ax=axes[1], palette=['#e74c3c', '#f39c12', '#2ecc71'])
axes[1].set_title('Distribusi Kelas Sentimen', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentimen')
axes[1].set_ylabel('Jumlah Ulasan')
for i, v in enumerate(sent_counts.values):
    axes[1].text(i, v + (max(sent_counts.values) * 0.015), f"{v:,}\n({v/len(df_screened)*100:.1f}%)", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('distribusi_sentimen.png', dpi=300)
plt.show()


---
## 5. Pipeline Pre-Processing Teks (Dua Jalur: Klasik/TF-IDF & IndoBERT)
Perbaikan utama pada tahap ini:
* **Jalur (b) IndoBERT (`teks_indobert`):** Hanya *cleaning* ringan tanpa stopword removal dan tanpa stemming agar struktur kalimat tetap alami.
* **Emoji $\rightarrow$ Teks Sentimen:** Mengubah emoji (😡, 👍, 👎, 😭) ke kata emosi (`marah`, `bagus`, `buruk`, `sedih`) sebelum pembersihan non-ASCII.
* **Normalisasi Huruf Berulang `(.)\1{2,} -> \1`:** Menangani `"bagusss"`, `"mahalll"`, `"lemooott"` menjadi bentuk tunggal sebelum pencocokan slang.
* **Penanganan Kata Ulang & Tanda Hubung:** `"bagus-bagus"` / `"bagus2"` $\rightarrow$ `"bagus"`; `"tiba-tiba"` / `"tiba2"` dipertahankan sebagai `"tiba-tiba"` (bukan `"tibatiba"`).
* **Penggantian Frasa Berbasis Regex + Slang + Inggris:** `"force close"` diproses regex sebelum tokenizing; kata slang baru (`kaga/kagak`, `engga`, `gada`, `knapa`, `pakek/dipake`, `nelpon`, `nihh`, `ngelek`) dan kata Inggris bermakna (`good -> bagus`) dinormalisasi.
* **Negasi Berbasis Whitelist:** Hanya menggabungkan `negasi_kata` jika kata berikutnya ada dalam `NEGATION_TARGET_WHITELIST`.
* **Stopword Removal Aman Kontras/Penguat:** Mempertahankan kata kontras (`tapi`, `padahal`, `namun`, ...) & penguat (`sangat`, `banget`, `kurang`, `sekali`), menampilkan daftar stopword final dengan penanda `[SENTIMEN]`, serta menyimpan teks pra-stopword berbatas klausa untuk n-gram.


In [ ]:
# 5.0 - 5.2: Jalur IndoBERT, Case Folding, Emoji, Huruf Berulang, Kata Ulang & Tanda Hubung
def convert_emojis_to_sentiment(text):
    # Alasan (A.6): Konversi emoji ke kata sentimen sebelum karakter non-ASCII dihapus
    for emj, sentiment_word in EMOJI_SENTIMENT_MAP.items():
        if emj in text:
            text = text.replace(emj, sentiment_word)
    if HAS_EMOJI_LIB:
        demojized = emoji.demojize(text, delimiters=(' __EMO_', '__ '))
        def _map_demoji(match):
            name = match.group(1).lower()
            if any(k in name for k in ('angry', 'rage', 'pouting', 'steam')):
                return ' marah '
            if any(k in name for k in ('cry', 'sob', 'sad', 'pensive', 'disappointed', 'pleading')):
                return ' sedih '
            if any(k in name for k in ('thumbs_down', 'pile_of_poo', 'nauseated', 'vomiting')):
                return ' buruk '
            if any(k in name for k in ('thumbs_up', 'star', 'check')):
                return ' bagus '
            if any(k in name for k in ('smile', 'grin', 'happy', 'joy', 'laugh')):
                return ' senang '
            if any(k in name for k in ('heart', 'love')):
                return ' suka '
            if any(k in name for k in ('folded_hands', 'pray')):
                return ' mohon '
            return ' '
        text = re.sub(r'__EMO_([a-zA-Z0-9_&-]+)__', _map_demoji, demojized)
    return text


def normalize_reduplication_and_hyphens(text):
    # Alasan (A.4): Samakan superscript '²' ke '2', pertahankan 'tiba-tiba', dan satukan kata ulang
    text = text.replace('\u00b2', '2')
    idiom_redup_map = {
        r'\btiba\s*2(?:nya)?\b': 'tiba-tiba',
        r'\btibatiba\b': 'tiba-tiba',
        r'\bgara\s*2(?:nya)?\b': 'gara-gara',
        r'\bgaragara\b': 'gara-gara',
        r'\bsia\s*2\b': 'sia-sia',
        r'\bsiasia\b': 'sia-sia',
        r'\brata\s*2(?:nya)?\b': 'rata-rata',
        r'\bratarata\b': 'rata-rata',
        r'\bmasing\s*2\b': 'masing-masing',
        r'\bsehari\s*2\b': 'sehari-hari',
    }
    for pat, repl in idiom_redup_map.items():
        text = re.sub(pat, repl, text)

    # Lindungi kata leksikal bertanda hubung SEBELUM aturan 'X-X -> X' agar 'tiba-tiba' utuh
    protected = {}
    for idx, hw in enumerate(LEXICALIZED_HYPHEN_WORDS):
        if hw in text:
            key = f'__HYPH{idx}__'
            protected[key] = hw
            text = re.sub(rf'\b{re.escape(hw)}\b', key, text)

    text = re.sub(r'\b([a-z]{2,})2(?:nya|an)?\b', r'\1', text)
    text = re.sub(r'\b([a-z]{2,})-\1\b', r'\1', text)

    # Tanda hubung selain kata leksikal diubah ke spasi (bukan dihapus menjadi kata dempet)
    text = re.sub(r'(?<=[a-z])-(?=[a-z])', ' ', text)
    punct_no_underscore = string.punctuation.replace('_', '')
    text = text.translate(str.maketrans('', '', punct_no_underscore))

    for key, hw in protected.items():
        text = text.replace(key, hw)
    return text


def clean_for_indobert(text):
    # Alasan (D.4): Cleaning ringan untuk IndoBERT tanpa stopword removal dan tanpa stemming
    t = str(text).lower().replace('\u00b2', '2')
    t = convert_emojis_to_sentiment(t)
    t = re.sub(r'http\S+|www\S+|https\S+', '', t, flags=re.MULTILINE)
    t = re.sub(r'[@#]\w+', '', t)
    t = re.sub(r'(.)\1{2,}', r'\1', t)
    t = re.sub(r'[^\x00-\x7F]+', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()


def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[@#]\w+', '', text)
    text = convert_emojis_to_sentiment(text)
    text = normalize_reduplication_and_hyphens(text)
    # Alasan (A.3): Normalisasi >=3 huruf berulang menjadi 1 huruf sebelum pencocokan slang
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


df_processed = df_screened.copy()
df_processed['teks_indobert'] = df_processed['content'].apply(clean_for_indobert)
df_processed['case_folded'] = df_processed['content'].apply(lambda x: str(x).lower())
df_processed['cleaned'] = df_processed['case_folded'].apply(clean_text)

print("✅ Jalur IndoBERT, Case Folding & Text Cleaning selesai!")
df_processed[['content', 'cleaned', 'teks_indobert']].head(3)


In [ ]:
# 5.3 & 5.4: Penggantian Frasa (Regex), Normalisasi Slang/Inggris, & Negasi Whitelist
compiled_phrases = [
    (re.compile(rf'\b{re.escape(phrase)}\b'), replacement)
    for phrase, replacement in PHRASE_REPLACEMENTS.items()
]

def normalize_slang_and_phrases(text):
    # Alasan (B.2): Ganti entri berspasi (mis. 'force close') dengan regex sebelum tokenizing
    for pat, repl in compiled_phrases:
        text = pat.sub(repl, text)
    words = text.split()
    normalized = []
    for w in words:
        mapped = SLANG_DICT.get(w, ENGLISH_TO_INDONESIAN.get(w, w))
        if mapped:  # Abaikan token yang sengaja dipetakan ke "" (mis. 'nihh')
            normalized.append(mapped)
    return ' '.join(normalized)


def handle_negation(text):
    # Alasan (A.5): Gabungkan negasi HANYA jika kata berikutnya masuk NEGATION_TARGET_WHITELIST
    words = text.split()
    result = []
    i = 0
    while i < len(words):
        if words[i] in NEGATION_WORDS and i + 1 < len(words) and words[i + 1] in NEGATION_TARGET_WHITELIST:
            result.append(f"{words[i]}_{words[i + 1]}")
            i += 2
        else:
            result.append(words[i])
            i += 1
    return ' '.join(result)


df_processed['slang_normalized'] = df_processed['cleaned'].apply(normalize_slang_and_phrases)
df_processed['negation_handled'] = df_processed['slang_normalized'].apply(handle_negation)

print("✅ Normalisasi frasa/slang & negasi berbasis whitelist selesai!")
df_processed[['cleaned', 'slang_normalized', 'negation_handled']].head(3)


In [ ]:
# 5.5: Tokenizing & Stopword Removal (Pertahankan Kata Kontras/Penguat & Tampilkan Daftar Final)
nltk_stops = set(stopwords.words('indonesian'))
sastrawi_stops = set(StopWordRemoverFactory().get_stop_words())
custom_stops = {
    'telkomsel', 'mytelkomsel', 'aplikasi', 'app', 'apk', 'telkom',
    'nya', 'nih', 'nihh', 'sih', 'deh', 'dong', 'kan', 'tuh', 'lah', 'loh', 'woi',
    'kak', 'min', 'admin', 'bintang', 'kasih',
    # Alasan (A.2): 'banget', 'sangat', 'sekali' dihapus dari daftar stopword
    'kali', 'versi', 'bisa', 'ada', 'dari', 'yang', 'untuk',
    'dengan', 'ini', 'itu', 'di', 'ke', 'pada', 'dan', 'atau',
    'saya', 'kamu', 'kami', 'kita', 'mereka', 'dia', 'kau', 'nge',
    'dah', 'udah', 'udh', 'gitu', 'gito', 'cuman', 'cuma', 'doang',
    'mending', 'kayak', 'kaya', 'biar', 'biarin', 'pas', 'gin', 'begini',
    'ampun', 'mulu', 'terus', 'kesini', 'dikit', 'sedikit',
    'gimana', 'bngt', 'yaa', 'yah', 'ngak', 'nggak',
    'kadang', 'kadangan', 'emang', 'memang', 'aja', 'saja',
    'juga', 'masih', 'sama', 'lagi', 'pernah', 'sekarang',
    'coba', 'pilih', 'bikin', 'main', 'mah', 'suruh', 'maaf',
    'ribu', 'jam', 'minggu', 'kota', 'terima', 'semoga', 'moga',
    'the', 'menit', 'tanggal', 'malam', 'kemarin', 'belah', 'gilir',
    'pokok', 'ulang', 'ubah', 'tinggal', 'bilang', 'beda', 'atur',
    'minimal', 'full', 'sengaja', 'utama', 'butuh', 'alih', 'lihat',
    'dapat', 'hadehh', 'bos', 'mas', 'kah', 'tidak_seperti',
    'jangan_mahal', 'info', 'hasil', 'proses', 'tawar', 'batas',
    'usaha', 'muncul', 'ekonomi', 'indonesia', 'konsumen',
    'belum_ada', 'tidak_punya', 'tidak_di', 'tidak_ngotak',
}

all_stops = nltk_stops.union(sastrawi_stops).union(ENGLISH_BASIC_STOPWORDS).union(custom_stops)
# Alasan (A.2): Pastikan kata negasi, kontras, dan penguat TIDAK dihapus sebagai stopword
all_stops -= NEGATION_WORDS
all_stops -= CONTRAST_AND_INTENSIFIER_WORDS

# Alasan (A.2): Tampilkan daftar stopword final dan tandai kata bernuansa sentimen
sorted_stops = sorted(all_stops)
stop_records = []
for w in sorted_stops:
    sumber = []
    if w in nltk_stops: sumber.append('NLTK')
    if w in sastrawi_stops: sumber.append('Sastrawi')
    if w in ENGLISH_BASIC_STOPWORDS: sumber.append('English')
    if w in custom_stops: sumber.append('Custom')
    has_sent = w in SENTIMENT_FLAGGED_STOPS
    stop_records.append({
        'kata': w, 'sumber': '+'.join(sumber),
        'bermakna_sentimen': has_sent,
        'penanda': '[SENTIMEN]' if has_sent else '',
    })

df_stops = pd.DataFrame(stop_records)
df_stops.to_csv('daftar_stopword_final.csv', index=False, encoding='utf-8')
print(f"📋 Total Stopword Final: {len(sorted_stops)} kata (tersimpan di 'daftar_stopword_final.csv').")
print("   Kata stopword bertanda [SENTIMEN]:", [r['kata'] for r in stop_records if r['bermakna_sentimen']])
print("   Daftar Stopword Final:")
print(', '.join(f"{r['kata']}{r['penanda']}" for r in stop_records))


def tokenize_preserve_hyphen(text):
    return [tok for tok in word_tokenize(text) if re.search(r'[a-z]', tok)]


df_processed['pre_stop_tokens'] = df_processed['negation_handled'].apply(tokenize_preserve_hyphen)
df_processed['tokens'] = df_processed['pre_stop_tokens'].apply(
    lambda toks: [w for w in toks if w not in all_stops and len(w) > 2]
)
df_processed['stopwords_removed'] = df_processed['tokens'].apply(lambda x: ' '.join(x))
print("\n✅ Tokenizing & Stopword Removal selesai!")


In [ ]:
# 5.6 & 5.7: Stemming Sastrawi (Unique Word Caching), N-Gram Pra-Stopword, OOV Top 100 & Ekspor 2 Jalur
stemmer = StemmerFactory().create_stemmer()
unique_tokens = set()
for t_list in df_processed['pre_stop_tokens']:
    unique_tokens.update(t_list)

print(f"💡 Jumlah kata unik (pra-stopword) yang di-stem: {len(unique_tokens):,}")
start_stem = time.time()
stem_cache = {}
for word in tqdm(unique_tokens, desc="Stemming Vocabulary Unik"):
    if word in STEM_EXCEPTIONS:
        stem_cache[word] = word
    elif '_' in word:
        parts = word.split('_', 1)
        base = parts[1]
        stem_cache[word] = word if base in STEM_EXCEPTIONS else f"{parts[0]}_{stemmer.stem(base)}"
    else:
        stem_cache[word] = stemmer.stem(word)
print(f"⚡ Stemming selesai dalam {time.time() - start_stem:.1f} detik!")


def apply_cache_post_stop(tokens):
    return [stem_cache.get(w, w) for w in tokens if stem_cache.get(w, w) not in all_stops and len(stem_cache.get(w, w)) > 2]


def build_clause_marked_pre_stop(pre_tokens):
    # Alasan (A.1): Pisahkan klausa pada kata kontras (' | ') SEBELUM stopword removal
    # agar n-gram tidak membentuk bigram semu lintas klausa.
    clauses, current = [], []
    for w in pre_tokens:
        root = stem_cache.get(w, w)
        if root in CONTRAST_WORDS:
            if current:
                clauses.append(' '.join(current))
                current = []
        elif len(root) > 1:
            current.append(root)
    if current:
        clauses.append(' '.join(current))
    return ' | '.join(clauses)


df_processed['stemmed_tokens'] = df_processed['tokens'].apply(apply_cache_post_stop)
df_processed['teks_final'] = df_processed['stemmed_tokens'].apply(lambda x: ' '.join(x))
df_processed['teks_klasik'] = df_processed['teks_final']
df_processed['teks_ngram_pre_stopword'] = df_processed['pre_stop_tokens'].apply(build_clause_marked_pre_stop)

# Alasan (B.4): Cari kandidat slang baru (OOV terhadap kamus kata dasar Sastrawi)
sastrawi_vocab = getattr(getattr(stemmer, 'dictionary', None), 'words', set())
known_vocab = set(sastrawi_vocab).union(SLANG_DICT.keys()).union(SLANG_DICT.values()).union(all_stops)
oov_counter = Counter()
oov_examples = {}
for raw_text, pre_toks in zip(df_processed['content'], df_processed['pre_stop_tokens']):
    for tok in pre_toks:
        root_tok = stem_cache.get(tok, tok)
        if len(tok) >= 2 and '_' not in tok and '-' not in tok and tok not in known_vocab and root_tok not in sastrawi_vocab:
            oov_counter[tok] += 1
            if tok not in oov_examples:
                oov_examples[tok] = str(raw_text).replace('\n', ' ')[:100]

df_oov_top100 = pd.DataFrame([
    {'Rank': i + 1, 'Token_OOV': tok, 'Frekuensi': cnt, 'Contoh_Konteks': oov_examples.get(tok, '')}
    for i, (tok, cnt) in enumerate(oov_counter.most_common(100))
])
df_oov_top100.to_csv('kandidat_slang_top100.csv', index=False, encoding='utf-8')
print("📋 Top 100 kandidat slang/OOV untuk tinjauan manual (tersimpan di 'kandidat_slang_top100.csv'):")
display(df_oov_top100.head(30))

df_final = df_processed[df_processed['teks_final'].str.strip() != ''].copy().reset_index(drop=True)
df_final.to_csv('ulasan_terproses.csv', index=False, encoding='utf-8')

# Alasan (D.4): Simpan dua jalur output terpisah untuk TF-IDF/klasik vs IndoBERT
cols_meta = [c for c in ['at', 'tanggal_ulasan', 'appVersion', 'score', 'sentiment'] if c in df_final.columns]
df_final[cols_meta + ['teks_klasik']].to_csv('dataset_klasik_tfidf.csv', index=False, encoding='utf-8')
df_final[cols_meta + ['teks_indobert']].to_csv('dataset_indobert.csv', index=False, encoding='utf-8')
print(f"✅ Pre-processing selesai! {len(df_final):,} ulasan valid siap dianalisis.")


---
## 6. Validasi Kualitas Label Rating
1. **Audit Manual 150 Ulasan Rating 4–5 Bermuatan Kata Negatif:** Banyak pengguna memberi rating tinggi agar ulasan mereka dibaca atau menulis pujian disertai keluhan (*mixed review*).
2. **Validasi Silang dengan `sentimen_leksikon`:** Mengukur seberapa besar ketidaksesuaian antara label berbasis rating dengan polaritas kata dalam teks, serta mengevaluasi opsi penghapusan kelas **Netral (rating 3)**.


In [ ]:
# 6.1 Sampel acak 150 ulasan rating 4-5 dengan kata negatif
neg_seed_words = {
    'mahal', 'lambat', 'lemot', 'lelet', 'buruk', 'jelek', 'tidak_bisa',
    'error', 'bug', 'lag', 'gagal', 'kecewa', 'parah', 'payah', 'susah',
    'sulit', 'sedot', 'hilang', 'rusak', 'boros', 'rugi', 'keluar_paksa',
}
raw_neg_pattern = re.compile(
    r'\b(?:' + '|'.join(re.escape(w.replace('_', ' ')) for w in neg_seed_words) + r')\b',
    re.IGNORECASE,
)

def _has_neg_signal(row):
    toks = set(str(row.get('negation_handled', '')).split()) | set(str(row.get('teks_final', '')).split())
    return bool(toks & neg_seed_words) or bool(raw_neg_pattern.search(str(row.get('content', ''))))

df_r45_neg = df_final[df_final['score'].isin([4, 5]) & df_final.apply(_has_neg_signal, axis=1)].copy()
df_audit = df_r45_neg.sample(n=min(150, len(df_r45_neg)), random_state=42)[['content', 'score']].reset_index(drop=True)
df_audit.to_csv('audit_rating45_negatif_150.csv', index=False, encoding='utf-8')

print(f"🔎 Ditemukan {len(df_r45_neg):,} ulasan rating 4-5 mengandung kata negatif. Menampilkan sampel 150 ulasan (disimpan ke 'audit_rating45_negatif_150.csv'):")
with pd.option_context('display.max_rows', 150, 'display.max_colwidth', 140):
    display(df_audit)

# 6.2 Kolom sentimen_leksikon & tingkat ketidaksesuaian
def compute_lexicon_sentiment(text):
    words = str(text).split()
    score_lex = 0
    i = 0
    while i < len(words):
        w = words[i]
        if w in NEGATION_WORDS and i + 1 < len(words):
            nxt = words[i + 1]
            if nxt in POSITIVE_LEXICON:
                score_lex -= 1; i += 2; continue
            if nxt in NEGATIVE_LEXICON:
                score_lex += 1; i += 2; continue
        if w in POSITIVE_LEXICON:
            score_lex += 1
        elif w in NEGATIVE_LEXICON:
            score_lex -= 1
        i += 1
    return 'Positif' if score_lex > 0 else ('Negatif' if score_lex < 0 else 'Netral')

df_final['sentimen_leksikon'] = df_final['negation_handled'].apply(compute_lexicon_sentiment)
mismatch_pct = (df_final['sentiment'] != df_final['sentimen_leksikon']).mean() * 100
r45_neg_pct = (df_final.loc[df_final['score'].isin([4, 5]), 'sentimen_leksikon'] == 'Negatif').mean() * 100
r12_pos_pct = (df_final.loc[df_final['score'].isin([1, 2]), 'sentimen_leksikon'] == 'Positif').mean() * 100

print(f"\n📊 Tingkat Ketidaksesuaian Rating vs Sentimen Leksikon: {mismatch_pct:.2f}%")
print(f"   - Rating 4-5 (Positif) diprediksi Negatif oleh leksikon : {r45_neg_pct:.2f}%")
print(f"   - Rating 1-2 (Negatif) diprediksi Positif oleh leksikon : {r12_pos_pct:.2f}%")
display(pd.crosstab(df_final['sentiment'], df_final['sentimen_leksikon'], margins=True))

# Perbarui file output dengan sentimen_leksikon
df_final.to_csv('ulasan_terproses.csv', index=False, encoding='utf-8')
cols_meta = [c for c in ['at', 'tanggal_ulasan', 'appVersion', 'score', 'sentiment', 'sentimen_leksikon'] if c in df_final.columns]
df_final[cols_meta + ['teks_klasik']].to_csv('dataset_klasik_tfidf.csv', index=False, encoding='utf-8')
df_final[cols_meta + ['teks_indobert']].to_csv('dataset_indobert.csv', index=False, encoding='utf-8')


### Catatan Keterbatasan Pelabelan Berbasis Rating & Opsi Menghapus Kelas Netral (Rating 3)
* **Keterbatasan Proxy Rating:** Skor bintang Google Play Store sering tidak konsisten dengan isi teks. Sebagian pengguna memberi ★4–5 agar keluhannya dibaca tim pengembang (*"sengaja kasih bintang 5 biar dibaca..."*) atau memuji satu hal sambil mengeluhkan harga/sinyal. Sebaliknya, ulasan ★1–2 terkadang memakai sarkasme (*"mantap banget sinyalnya sampai gak bisa buka"*).
* **Opsi Menghapus Kelas Netral (Rating 3):** Proporsi kelas Netral hanya sekitar ~5,8% dan isinya sangat sering bercampur antara pujian dan kritik (*mixed sentiment*). Jika tujuan utama adalah klasifikasi polaritas yang tegas, menghapus rating 3 (menjadi biner Positif vs Negatif) dapat mengurangi *label noise*. Namun jika kelas Netral tetap dipertahankan untuk menangkap *mixed feedback*, evaluasi wajib memakai **macro-F1** serta pembobotan kelas (`class_weight`).


---
## 7. Analisis Frekuensi, N-gram Pra-Stopword, Log-Odds Ratio, dan Analisis Aspek
1. **Unigram, Bigram & Trigram:** Bigram dan trigram dibangun dari teks **sebelum stopword removal** yang dipecah pada kata kontras (`tapi`, `padahal`, `namun`, `sedangkan`, `kecuali`) sehingga tidak muncul bigram semu seperti `"mahal sinyal"`.
2. **Kata Diskriminatif per Kelas (Log-Odds Ratio Dirichlet Prior):** Menemukan top 20 kata pembeda khas masing-masing kelas sentimen.
3. **Analisis Aspek Berbasis Kata Kunci:** Mengukur proporsi ulasan dan sentimen pada aspek **Aplikasi**, **Jaringan**, **Harga/Paket**, dan **Layanan**.


In [ ]:
def extract_top_unigrams(corpus, top_n=20):
    words = []
    for doc in corpus:
        words.extend(str(doc).split())
    return Counter(words).most_common(top_n)

def get_top_ngrams_from_clauses(corpus_clauses, ngram_range, top_n=20):
    # Alasan (A.1): Pecah ulasan per klausa (' | ') agar tidak terbentuk n-gram lintas kata kontras
    clauses = []
    for doc in corpus_clauses.dropna():
        for seg in str(doc).split(' | '):
            seg = seg.strip()
            if len(seg.split()) >= ngram_range[0]:
                clauses.append(seg)
    if not clauses:
        return []
    vec = CountVectorizer(ngram_range=ngram_range, token_pattern=r'(?u)\b[a-zA-Z0-9_-]{2,}\b')
    bag = vec.fit_transform(clauses)
    sum_words = bag.sum(axis=0)
    words_freq = [(word, int(sum_words[0, idx])) for word, idx in vec.vocabulary_.items()]
    words_freq.sort(key=lambda x: x[1], reverse=True)
    return words_freq[:top_n]

def pad_list(items, length):
    return (items + [('', 0)] * (length - len(items)))[:length]

corpus_all = df_final['teks_final']
corpus_pos = df_final[df_final['sentiment'] == 'Positif']['teks_final']
corpus_neg = df_final[df_final['sentiment'] == 'Negatif']['teks_final']
corpus_net = df_final[df_final['sentiment'] == 'Netral']['teks_final']

top20_all = extract_top_unigrams(corpus_all, 20)
top20_pos = extract_top_unigrams(corpus_pos, 20)
top20_neg = extract_top_unigrams(corpus_neg, 20)

df_top20 = pd.DataFrame({
    'Rank': range(1, 21),
    'Kata (Semua)': [w[0] for w in top20_all], 'Frekuensi (Semua)': [w[1] for w in top20_all],
    'Kata (Positif)': [w[0] for w in top20_pos], 'Frekuensi (Positif)': [w[1] for w in top20_pos],
    'Kata (Negatif)': [w[0] for w in top20_neg], 'Frekuensi (Negatif)': [w[1] for w in top20_neg],
})
df_top20.to_csv('top20_unigrams_per_sentimen.csv', index=False, encoding='utf-8')

ng_all = df_final['teks_ngram_pre_stopword']
ng_pos = df_final[df_final['sentiment'] == 'Positif']['teks_ngram_pre_stopword']
ng_neg = df_final[df_final['sentiment'] == 'Negatif']['teks_ngram_pre_stopword']

top20_bi_all = get_top_ngrams_from_clauses(ng_all, (2, 2), 20)
top20_bi_pos = get_top_ngrams_from_clauses(ng_pos, (2, 2), 20)
top20_bi_neg = get_top_ngrams_from_clauses(ng_neg, (2, 2), 20)

top20_tri_all = get_top_ngrams_from_clauses(ng_all, (3, 3), 20)
top20_tri_pos = get_top_ngrams_from_clauses(ng_pos, (3, 3), 20)
top20_tri_neg = get_top_ngrams_from_clauses(ng_neg, (3, 3), 20)

bi_all_p, bi_pos_p, bi_neg_p = pad_list(top20_bi_all, 20), pad_list(top20_bi_pos, 20), pad_list(top20_bi_neg, 20)
df_bigram = pd.DataFrame({
    'Rank': range(1, 21),
    'Bigram (Semua)': [w[0] for w in bi_all_p], 'Frekuensi (Semua)': [w[1] for w in bi_all_p],
    'Bigram (Positif)': [w[0] for w in bi_pos_p], 'Frekuensi (Positif)': [w[1] for w in bi_pos_p],
    'Bigram (Negatif)': [w[0] for w in bi_neg_p], 'Frekuensi (Negatif)': [w[1] for w in bi_neg_p],
})
df_bigram.to_csv('top20_bigrams_per_sentimen.csv', index=False, encoding='utf-8')

tri_all_p, tri_pos_p, tri_neg_p = pad_list(top20_tri_all, 20), pad_list(top20_tri_pos, 20), pad_list(top20_tri_neg, 20)
df_trigram = pd.DataFrame({
    'Rank': range(1, 21),
    'Trigram (Semua)': [w[0] for w in tri_all_p], 'Frekuensi (Semua)': [w[1] for w in tri_all_p],
    'Trigram (Positif)': [w[0] for w in tri_pos_p], 'Frekuensi (Positif)': [w[1] for w in tri_pos_p],
    'Trigram (Negatif)': [w[0] for w in tri_neg_p], 'Frekuensi (Negatif)': [w[1] for w in tri_neg_p],
})
df_trigram.to_csv('top20_trigrams_per_sentimen.csv', index=False, encoding='utf-8')

def plot_bars(ax, item_list, title, palette, xlabel='Frekuensi'):
    words = [item[0] for item in item_list][::-1]
    vals = [item[1] for item in item_list][::-1]
    sns.barplot(x=vals, y=words, ax=ax, palette=palette)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel(xlabel)
    max_v = max(vals) if vals else 1
    for i, v in enumerate(vals):
        lbl = f"{v:,}" if isinstance(v, int) else f"{v:.2f}"
        ax.text(v + (max_v * 0.015), i, lbl, va='center', fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
plot_bars(axes[0], top20_all, 'Top 20 Unigram: Keseluruhan', 'Blues_d')
plot_bars(axes[1], top20_pos, 'Top 20 Unigram: Positif (Rating 4-5)', 'Greens_d')
plot_bars(axes[2], top20_neg, 'Top 20 Unigram: Negatif (Rating 1-2)', 'Reds_d')
plt.tight_layout(); plt.savefig('barchart_top20_kata.png', dpi=300); plt.show()

fig2, axes2 = plt.subplots(1, 3, figsize=(22, 8))
plot_bars(axes2[0], top20_bi_all[:15], 'Top 15 Bigram (Pra-Stopword): Keseluruhan', 'Blues_d')
plot_bars(axes2[1], top20_bi_pos[:15], 'Top 15 Bigram (Pra-Stopword): Positif', 'Greens_d')
plot_bars(axes2[2], top20_bi_neg[:15], 'Top 15 Bigram (Pra-Stopword): Negatif', 'Reds_d')
plt.tight_layout(); plt.savefig('barchart_top15_bigram.png', dpi=300); plt.show()


In [ ]:
# 7.3 Kata Diskriminatif per Kelas (Log-Odds Ratio dengan Informative Dirichlet Prior)
def compute_log_odds_dirichlet(corpus_target, corpus_other, min_total_freq=5, prior_alpha=0.01):
    # Alasan (D.2): Log-odds ratio berprior Dirichlet menyoroti kata paling pembeda tiap kelas
    c1, c2 = Counter(), Counter()
    for doc in corpus_target.dropna(): c1.update(str(doc).split())
    for doc in corpus_other.dropna(): c2.update(str(doc).split())
    total = c1 + c2
    vocab = [w for w, cnt in total.items() if cnt >= min_total_freq]
    if not vocab:
        return []
    n1, n2 = sum(c1[w] for w in vocab), sum(c2[w] for w in vocab)
    alpha_map = {w: max(prior_alpha, total[w] * 0.1) for w in vocab}
    alpha_0 = sum(alpha_map.values())
    scores = []
    for w in vocab:
        a_w = alpha_map[w]
        y1, y2 = c1[w] + a_w, c2[w] + a_w
        delta = np.log(y1 / max(1e-9, (n1 + alpha_0) - y1)) - np.log(y2 / max(1e-9, (n2 + alpha_0) - y2))
        sigma = np.sqrt((1.0 / y1) + (1.0 / y2))
        scores.append((w, float(delta / sigma), c1[w], c2[w]))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores

logodds_pos = compute_log_odds_dirichlet(corpus_pos, df_final[df_final['sentiment'] != 'Positif']['teks_final'])
logodds_net = compute_log_odds_dirichlet(corpus_net, df_final[df_final['sentiment'] != 'Netral']['teks_final'])
logodds_neg = compute_log_odds_dirichlet(corpus_neg, df_final[df_final['sentiment'] != 'Negatif']['teks_final'])

top20_lo_pos = pad_list([(w, round(z, 3)) for w, z, _, _ in logodds_pos[:20]], 20)
top20_lo_net = pad_list([(w, round(z, 3)) for w, z, _, _ in logodds_net[:20]], 20)
top20_lo_neg = pad_list([(w, round(z, 3)) for w, z, _, _ in logodds_neg[:20]], 20)

df_logodds = pd.DataFrame({
    'Rank': range(1, 21),
    'Kata_Positif': [w for w, _ in top20_lo_pos], 'Z_LogOdds_Positif': [z for _, z in top20_lo_pos],
    'Kata_Netral': [w for w, _ in top20_lo_net], 'Z_LogOdds_Netral': [z for _, z in top20_lo_net],
    'Kata_Negatif': [w for w, _ in top20_lo_neg], 'Z_LogOdds_Negatif': [z for _, z in top20_lo_neg],
})
df_logodds.to_csv('top20_kata_diskriminatif_logodds.csv', index=False, encoding='utf-8')
display(df_logodds)

fig_lo, axes_lo = plt.subplots(1, 3, figsize=(22, 8))
plot_bars(axes_lo[0], top20_lo_pos, 'Top 20 Kata Diskriminatif: Positif', 'Greens_d', xlabel='Z-score Log-Odds')
plot_bars(axes_lo[1], top20_lo_net, 'Top 20 Kata Diskriminatif: Netral', 'Oranges_d', xlabel='Z-score Log-Odds')
plot_bars(axes_lo[2], top20_lo_neg, 'Top 20 Kata Diskriminatif: Negatif', 'Reds_d', xlabel='Z-score Log-Odds')
plt.tight_layout(); plt.savefig('barchart_top20_diskriminatif_logodds.png', dpi=300); plt.show()


In [ ]:
# 7.4 Analisis Aspek Berbasis Kamus Kata Kunci
aspect_rows = []
total_docs = len(df_final)
for aspect_name, kw_set in ASPECT_KEYWORDS.items():
    has_aspect = df_final.apply(
        lambda r: bool((set(str(r.get('teks_final', '')).split()) | set(str(r.get('negation_handled', '')).split())) & kw_set),
        axis=1,
    )
    df_final[f'aspek_{aspect_name}'] = has_aspect
    sub = df_final[has_aspect]
    cnt = len(sub)
    sd = sub['sentiment'].value_counts().to_dict()
    aspect_rows.append({
        'Aspek': aspect_name, 'Jumlah_Ulasan': cnt,
        'Proporsi_Total_Pct': round(cnt / total_docs * 100, 2) if total_docs else 0.0,
        'Negatif': sd.get('Negatif', 0), 'Netral': sd.get('Netral', 0), 'Positif': sd.get('Positif', 0),
        'Negatif_Pct': round(sd.get('Negatif', 0) / cnt * 100, 2) if cnt else 0.0,
        'Netral_Pct': round(sd.get('Netral', 0) / cnt * 100, 2) if cnt else 0.0,
        'Positif_Pct': round(sd.get('Positif', 0) / cnt * 100, 2) if cnt else 0.0,
    })

df_aspect = pd.DataFrame(aspect_rows)
df_aspect.to_csv('analisis_aspek_sentimen.csv', index=False, encoding='utf-8')
display(df_aspect)

fig_asp, axes_asp = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x='Aspek', y='Proporsi_Total_Pct', data=df_aspect, ax=axes_asp[0], palette='Blues_d')
axes_asp[0].set_title('Proporsi Ulasan per Aspek (%)', fontsize=13, fontweight='bold')
axes_asp[0].set_ylabel('Persentase Ulasan (%)')
for i, row in df_aspect.iterrows():
    axes_asp[0].text(i, row['Proporsi_Total_Pct'] + 1.0, f"{row['Proporsi_Total_Pct']:.1f}%\n({row['Jumlah_Ulasan']:,})", ha='center', fontsize=9)

x_idx = np.arange(len(df_aspect))
neg_p, net_p, pos_p = df_aspect['Negatif_Pct'].values, df_aspect['Netral_Pct'].values, df_aspect['Positif_Pct'].values
axes_asp[1].bar(x_idx, neg_p, label='Negatif', color='#e74c3c')
axes_asp[1].bar(x_idx, net_p, bottom=neg_p, label='Netral', color='#f39c12')
axes_asp[1].bar(x_idx, pos_p, bottom=neg_p + net_p, label='Positif', color='#2ecc71')
axes_asp[1].set_xticks(x_idx); axes_asp[1].set_xticklabels(df_aspect['Aspek'])
axes_asp[1].set_ylim(0, 108); axes_asp[1].set_ylabel('Proporsi Sentimen (%)')
axes_asp[1].set_title('Distribusi Sentimen per Aspek (%)', fontsize=13, fontweight='bold')
axes_asp[1].legend(loc='upper right')
for i in range(len(df_aspect)):
    axes_asp[1].text(i, neg_p[i] / 2, f"{neg_p[i]:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    if pos_p[i] >= 8:
        axes_asp[1].text(i, neg_p[i] + net_p[i] + pos_p[i] / 2, f"{pos_p[i]:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=9)
plt.tight_layout(); plt.savefig('grafik_analisis_aspek.png', dpi=300); plt.show()


---
## 8. Visualisasi Word Cloud (Colormap Sedang-Gelap, Sensor Kata Kasar, & Versi Log-Odds)
Perbaikan visualisasi Word Cloud:
* Memakai `color_func` pada rentang `[0.45, 0.92]` dari colormap sekuensial sehingga kata berfrekuensi tinggi selalu tampil dalam warna **sedang-gelap** (tidak pernah pudar/terlalu terang).
* Menyensor kata kasar (`PROFANITY_FILTER`) **khusus pada gambar Word Cloud** tanpa mengubah isi dataset.
* Menambahkan Word Cloud berbobot **Log-Odds Ratio** untuk kelas Positif dan Negatif.


In [ ]:
def make_sequential_color_func(cmap_name, min_val=0.45, max_val=0.92):
    # Alasan (D.1): Mengunci rentang warna [0.45, 0.92] agar kata penting berwarna sedang-gelap
    cmap = plt.colormaps.get_cmap(cmap_name)
    def _color_func(word, font_size, position, orientation, random_state=None, **kwargs):
        norm = min(1.0, max(0.0, (font_size - 10) / 110.0))
        rgba = cmap(min_val + norm * (max_val - min_val))
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"
    return _color_func

def build_clean_wc_frequencies(corpus_text, exclude_words):
    freq = Counter()
    full_exclude = set(exclude_words).union(PROFANITY_FILTER)
    for doc in corpus_text.dropna():
        for w in str(doc).split():
            if w not in full_exclude:
                freq[w] += 1
    return freq

def make_wordcloud_from_freq(freq_dict, title, cmap_name, output_png):
    if not freq_dict:
        print(f"⚠️ Frekuensi kosong untuk '{output_png}'.")
        return
    wc = WordCloud(
        width=1200, height=600, background_color='white',
        color_func=make_sequential_color_func(cmap_name),
        max_words=120, random_state=42, collocations=False, normalize_plurals=False,
    ).generate_from_frequencies(freq_dict)
    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=15, fontweight='bold', pad=18)
    plt.tight_layout(pad=0)
    plt.savefig(output_png, dpi=300, bbox_inches='tight')
    plt.show()

make_wordcloud_from_freq(build_clean_wc_frequencies(corpus_all, set()), 'Word Cloud: Keseluruhan Ulasan MyTelkomsel', 'Blues', 'wordcloud_keseluruhan.png')
pos_exclude = NEGATIVE_LEXICON.union(NEUTRAL_TOPIC_WORDS).union(NEGATION_WORDS).union(CONTRAST_WORDS).union({'kurang', 'agak', 'terlalu', 'repot', 'tidak_tahu'})
make_wordcloud_from_freq(build_clean_wc_frequencies(corpus_pos, pos_exclude), 'Word Cloud: Ulasan Positif MyTelkomsel (Rating 4 - 5)', 'Greens', 'wordcloud_positif.png')
make_wordcloud_from_freq(build_clean_wc_frequencies(corpus_neg, POSITIVE_LEXICON.union(NEUTRAL_TOPIC_WORDS)), 'Word Cloud: Ulasan Negatif (Frekuensi, Disensor)', 'Reds', 'wordcloud_negatif.png')

lo_pos_weights = {w: z for w, z, _, _ in logodds_pos if z > 0 and w not in PROFANITY_FILTER}
lo_neg_weights = {w: z for w, z, _, _ in logodds_neg if z > 0 and w not in PROFANITY_FILTER}
make_wordcloud_from_freq(lo_pos_weights, 'Word Cloud Berbobot Log-Odds: Ulasan Positif', 'Greens', 'wordcloud_positif_logodds.png')
make_wordcloud_from_freq(lo_neg_weights, 'Word Cloud Berbobot Log-Odds: Ulasan Negatif', 'Reds', 'wordcloud_negatif_logodds.png')


---
## 9. Persiapan Dataset Modeling: Stratified Split 80/20, `class_weight`, & Metrik `macro-F1`
* **Split Stratified 80/20 (`random_state=42`):** Menyimpan `train_data.csv` dan `test_data.csv` dengan kolom numerik `label` (`0=Negatif, 1=Netral, 2=Positif`), mempertahankan `teks_klasik`, `teks_indobert`, `at`, dan `appVersion`.
* **Penanganan Ketidakseimbangan Kelas:** Menghitung `class_weight='balanced'` dari data **train**, serta mencontohkan oversampling **hanya pada data train** (tidak pernah pada data test untuk mencegah *data leakage*).
* **Mengapa `macro-F1` adalah Metrik Utama:** Dengan distribusi timpang (~78% Negatif, ~6% Netral, ~16% Positif), *Accuracy* sangat mudah menipu. `macro-F1` memberi bobot setara bagi setiap kelas sehingga kegagalan pada kelas minoritas tetap terukur jelas.


In [ ]:
label_map = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
df_model = df_final.copy()
df_model['label'] = df_model['sentiment'].map(label_map)

keep_cols = [c for c in ['at', 'tanggal_ulasan', 'appVersion', 'score', 'sentiment', 'label', 'sentimen_leksikon', 'teks_klasik', 'teks_indobert'] if c in df_model.columns]
df_model = df_model[keep_cols]

df_train, df_test = train_test_split(df_model, test_size=0.20, stratify=df_model['label'], random_state=42)
df_train, df_test = df_train.reset_index(drop=True), df_test.reset_index(drop=True)
df_train.to_csv('train_data.csv', index=False, encoding='utf-8')
df_test.to_csv('test_data.csv', index=False, encoding='utf-8')

classes = np.array([0, 1, 2])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=df_train['label'].values)
class_weight_dict = {int(c): round(float(w), 4) for c, w in zip(classes, weights)}

# Contoh oversampling HANYA pada data train
max_cnt = df_train['label'].value_counts().max()
df_train_oversampled = pd.concat([
    grp.sample(n=max_cnt, replace=True, random_state=42) for _, grp in df_train.groupby('label')
]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"✅ Stratified Split 80/20: Train = {len(df_train):,} | Test = {len(df_test):,}")
print("   Proporsi Train:", df_train['sentiment'].value_counts().to_dict())
print("   Proporsi Test :", df_test['sentiment'].value_counts().to_dict())
print(f"⚖️  Contoh class_weight ('balanced') untuk LogisticRegression/LinearSVC/CrossEntropyLoss: {class_weight_dict}")
print(f"🔁 Contoh oversampling (HANYA Train): {len(df_train):,} -> {len(df_train_oversampled):,} baris (Test tetap {len(df_test):,})")


---
## 10. Pengemasan Seluruh Output (Google Colab)
Mengemas seluruh file CSV, grafik, dan Word Cloud terbaru ke dalam `hasil_analisis_mytelkomsel.zip`.


In [ ]:
artifacts = [
    'ulasan_mentah.csv', 'ulasan_screening.csv', 'ringkasan_screening.csv',
    'ulasan_terproses.csv', 'dataset_klasik_tfidf.csv', 'dataset_indobert.csv',
    'train_data.csv', 'test_data.csv', 'daftar_stopword_final.csv',
    'kandidat_slang_top100.csv', 'audit_rating45_negatif_150.csv',
    'top20_unigrams_per_sentimen.csv', 'top20_bigrams_per_sentimen.csv',
    'top20_trigrams_per_sentimen.csv', 'top20_kata_diskriminatif_logodds.csv',
    'analisis_aspek_sentimen.csv',
    'distribusi_sentimen.png', 'barchart_top20_kata.png', 'barchart_top15_bigram.png',
    'barchart_top20_diskriminatif_logodds.png', 'grafik_analisis_aspek.png',
    'wordcloud_keseluruhan.png', 'wordcloud_positif.png', 'wordcloud_negatif.png',
    'wordcloud_positif_logodds.png', 'wordcloud_negatif_logodds.png',
]

zip_output_name = 'hasil_analisis_mytelkomsel.zip'
with zipfile.ZipFile(zip_output_name, 'w') as zipf:
    for file_path in artifacts:
        if os.path.exists(file_path):
            zipf.write(file_path)
            print(f"📦 Ditambahkan: {file_path}")

try:
    from google.colab import files
    files.download(zip_output_name)
except ImportError:
    print(f"ℹ️ Berkas arsip lokal siap: {zip_output_name}")
